# Day 6：手写 LLaVA 多模态实践（本周核心之二！）

> **目标**：从零实现完整的 LLaVA 多模态大模型架构——`VisionEncoder → MLP Projector → 多模态拼接 → LLM 自回归生成`；实现两阶段训练流程（Stage 1 预训练对齐 + Stage 2 指令微调）；在合成数据上跑通端到端训练与推理；实现图文问答 Demo。

---

## 实现路线图

```
Part 1: 配置与环境
  LLaVAConfig (dataclass) — 统一管理 Vision / Projector / LLM 超参数

Part 2: Vision Encoder
  复用 Day 3 的 ViT 实现，提取倒数第二层的 patch tokens (去掉 CLS)

Part 3: MLP Projector
  LLaVA-1.0: Linear(D_v, D_l)
  LLaVA-1.5: Linear(D_v, D_h) → GELU → Linear(D_h, D_l)

Part 4: 多模态输入拼接
  Vision tokens + Text tokens → 统一序列 → Loss mask

Part 5: 简化 LLM Decoder
  复用 LLaMA 风格 Decoder：RMSNorm + Causal Self-Attention + SwiGLU FFN

Part 6: 完整 LLaVA 模型
  VisionEncoder (冻结) + MLPProjector + LLM → 端到端前向传播

Part 7: 两阶段训练
  Stage 1: 冻结 ViT + LLM，只训练 Projector
  Stage 2: 冻结 ViT，训练 Projector + LLM

Part 8: 推理 — 图文问答 Demo
  图像编码 → 投影 → 拼接 → 自回归生成 → 解码

Part 9: 对比验证与总结
  Shape 检查 + 参数量统计 + 与 HuggingFace LLaVA 对比
```

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, List

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part 1：模型配置

LLaVA 需要统一管理三个组件的超参数：
- **Vision Encoder**: CLIP ViT（Day 3 已实现）
- **Projector**: 将视觉维度 $D_v$ 映射到语言维度 $D_l$
- **LLM**: LLaMA 风格 Decoder

教学配置使用小维度，便于单卡快速验证。

In [ ]:
@dataclass
class LLaVAConfig:
    """LLaVA 模型统一配置。默认值为教学用小模型。"""
    # === Vision Encoder (ViT) ===
    image_size: int = 224
    patch_size: int = 16
    num_channels: int = 3
    vision_hidden_size: int = 256    # D_v (CLIP ViT-L: 1024)
    vision_num_layers: int = 6       # (CLIP ViT-L: 24)
    vision_num_heads: int = 8
    vision_mlp_ratio: int = 4

    # === Projector ===
    projector_type: str = 'mlp'      # 'linear' (LLaVA-1.0) or 'mlp' (LLaVA-1.5)

    # === LLM (LLaMA-style Decoder) ===
    vocab_size: int = 32000          # LLaMA tokenizer
    llm_hidden_size: int = 512       # D_l (LLaMA-7B: 4096)
    llm_num_layers: int = 8          # (LLaMA-7B: 32)
    llm_num_heads: int = 8           # (LLaMA-7B: 32)
    llm_intermediate_size: int = 1024 # SwiGLU FFN (LLaMA-7B: 11008)
    max_seq_len: int = 512           # (LLaMA: 2048+)
    rms_norm_eps: float = 1e-6
    dropout: float = 0.1

    # === Special tokens ===
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2
    image_token_id: int = 32000      # <image> 占位符

    @property
    def num_patches(self):
        return (self.image_size // self.patch_size) ** 2  # 196 for 224/16

    @property
    def vision_head_dim(self):
        return self.vision_hidden_size // self.vision_num_heads

    @property
    def llm_head_dim(self):
        return self.llm_hidden_size // self.llm_num_heads


config = LLaVAConfig()
print(f"Vision: {config.num_patches} patches, D_v={config.vision_hidden_size}")
print(f"LLM: D_l={config.llm_hidden_size}, {config.llm_num_layers} layers")
print(f"Projector: {config.projector_type}, {config.vision_hidden_size} → {config.llm_hidden_size}")

## Part 2：Vision Encoder — 复用 ViT 提取视觉特征

LLaVA 使用 CLIP ViT 作为视觉编码器，关键区别：
1. **使用倒数第二层输出**：保留更多空间信息
2. **去掉 CLS token**：只保留 patch tokens 作为视觉特征
3. **全程冻结**：ViT 参数不参与训练

$$\mathbf{Z}_v = \text{ViT}_{[-2]}(\text{Image}) \in \mathbb{R}^{N \times D_v}$$

其中 $N = (H/P)^2$ 是 patch 数，$[-2]$ 表示倒数第二层。

In [ ]:
class PatchEmbedding(nn.Module):
    """将图像切分为 patch 并投影，添加 CLS token 和位置编码。"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        self.num_patches = config.num_patches

        self.projection = nn.Conv2d(
            config.num_channels, config.vision_hidden_size,
            kernel_size=config.patch_size, stride=config.patch_size
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.vision_hidden_size) * 0.02)
        self.position_embedding = nn.Parameter(
            torch.randn(1, config.num_patches + 1, config.vision_hidden_size) * 0.02
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        # (B, C, H, W) → (B, D_v, H/P, W/P) → (B, D_v, N) → (B, N, D_v)
        x = self.projection(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, N+1, D_v)
        x = x + self.position_embedding
        return x


class ViTBlock(nn.Module):
    """ViT Encoder Block: Pre-LN + MSA + Residual + Pre-LN + FFN + Residual"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        D = config.vision_hidden_size
        self.ln1 = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, config.vision_num_heads, dropout=config.dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(D)
        self.mlp = nn.Sequential(
            nn.Linear(D, D * config.vision_mlp_ratio),
            nn.GELU(),
            nn.Linear(D * config.vision_mlp_ratio, D),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.mlp(self.ln2(x))
        return x


class VisionEncoder(nn.Module):
    """CLIP ViT Vision Encoder，输出倒数第二层的 patch tokens（去掉 CLS）。"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        self.patch_embed = PatchEmbedding(config)
        self.layers = nn.ModuleList([ViTBlock(config) for _ in range(config.vision_num_layers)])
        self.ln = nn.LayerNorm(config.vision_hidden_size)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(pixel_values)  # (B, N+1, D_v)

        # 通过所有层，但返回倒数第二层的输出
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i == len(self.layers) - 2:
                # 倒数第二层输出，去掉 CLS token
                hidden_states = self.ln(x[:, 1:, :])  # (B, N, D_v)

        return hidden_states

In [ ]:
# 验证 Vision Encoder
vision_encoder = VisionEncoder(config).to(device)
dummy_image = torch.randn(2, 3, config.image_size, config.image_size, device=device)
vision_out = vision_encoder(dummy_image)
print(f"Input:  {dummy_image.shape}")
print(f"Output: {vision_out.shape}")
print(f"Expected: (2, {config.num_patches}, {config.vision_hidden_size})")
assert vision_out.shape == (2, config.num_patches, config.vision_hidden_size)
print("✓ Vision Encoder shape 验证通过！")

## Part 3：MLP Projector — 视觉到语言的桥梁

Projector 是 LLaVA 架构中最核心的创新组件：用极少参数将视觉特征"翻译"到语言空间。

**LLaVA-1.0**（线性投影）：
$$\mathbf{H}_v = \mathbf{Z}_v \mathbf{W}, \quad \mathbf{W} \in \mathbb{R}^{D_v \times D_l}$$

**LLaVA-1.5**（两层 MLP）：
$$\mathbf{H}_v = \text{GELU}(\mathbf{Z}_v \mathbf{W}_1) \mathbf{W}_2$$
$$\mathbf{W}_1 \in \mathbb{R}^{D_v \times D_l}, \quad \mathbf{W}_2 \in \mathbb{R}^{D_l \times D_l}$$

### 面试高频：为什么简单的 MLP 就能 work？

1. CLIP 已经完成了视觉-语言语义对齐，Projector 只需做"维度适配"
2. LLM 的表示空间足够强大，可以"吸收"不完美的投影
3. 两阶段训练策略让 Projector 在 Stage 1 专注于对齐学习

In [ ]:
class MultimodalProjector(nn.Module):
    """LLaVA Projector: 将视觉特征投影到 LLM 的嵌入空间。
    
    支持两种模式:
    - 'linear': LLaVA-1.0, 单层线性投影
    - 'mlp':    LLaVA-1.5, 两层 MLP + GELU 激活
    """

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        D_v = config.vision_hidden_size
        D_l = config.llm_hidden_size

        if config.projector_type == 'linear':
            self.projector = nn.Linear(D_v, D_l)
        elif config.projector_type == 'mlp':
            self.projector = nn.Sequential(
                nn.Linear(D_v, D_l),
                nn.GELU(),
                nn.Linear(D_l, D_l),
            )
        else:
            raise ValueError(f"Unknown projector type: {config.projector_type}")

    def forward(self, vision_features: torch.Tensor) -> torch.Tensor:
        """(B, N, D_v) → (B, N, D_l)"""
        return self.projector(vision_features)

In [ ]:
# 验证 Projector
projector = MultimodalProjector(config).to(device)
proj_out = projector(vision_out)
print(f"Input:  {vision_out.shape}  (B, N, D_v={config.vision_hidden_size})")
print(f"Output: {proj_out.shape}  (B, N, D_l={config.llm_hidden_size})")
assert proj_out.shape == (2, config.num_patches, config.llm_hidden_size)
print("✓ MLP Projector shape 验证通过！")

# 参数量对比
linear_proj = MultimodalProjector(LLaVAConfig(projector_type='linear'))
mlp_proj = MultimodalProjector(LLaVAConfig(projector_type='mlp'))
linear_params = sum(p.numel() for p in linear_proj.parameters())
mlp_params = sum(p.numel() for p in mlp_proj.parameters())
print(f"\nLinear Projector 参数量: {linear_params:,} ({linear_params/1e6:.2f}M)")
print(f"MLP Projector 参数量:    {mlp_params:,} ({mlp_params/1e6:.2f}M)")

## Part 4：多模态输入拼接 — 让 LLM "看到" 图像

LLaVA 的核心设计：将视觉 token 和文本 token 在序列维度拼接，送入 LLM。

```
Token sequence:
  [VIS][VIS]...[VIS]  [BOS] What is this? [SEP] This is a cat. [EOS]
  ├── N 个视觉 token ──┤├── 问题 (human) ─┤├── 回答 (assistant) ─┤

Loss mask:
  [ 0   0  ...  0       0    0   0   0   0    1    1  1 1  1     1  ]
  ├── 不计算 loss ───────────────────────────┤├── 计算 loss ────────┤
```

### 关键实现细节
1. 视觉 token 已经通过 Projector 映射到 $D_l$ 维，可以直接与文本 embedding 拼接
2. **Loss 只计算 assistant 回答部分**：防止模型学习"复述问题"或"重建图像"
3. Causal mask 确保 LLM 只能看到前面的 token

In [ ]:
def prepare_multimodal_inputs(
    visual_tokens: torch.Tensor,
    input_ids: torch.Tensor,
    token_embedding: nn.Embedding,
    image_token_id: int = 32000,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    将视觉 token 插入到文本序列中 <image> 占位符的位置。
    
    Args:
        visual_tokens: (B, N_vis, D_l) 视觉 token
        input_ids: (B, T) 文本 token ids, 其中 image_token_id 标记图像位置
        token_embedding: LLM 的 token embedding 层
        image_token_id: <image> 占位符的 token id
    
    Returns:
        inputs_embeds: (B, T_new, D_l) 拼接后的 embedding
        attention_mask: (B, T_new) 注意力掩码
    """
    B, T = input_ids.shape
    N_vis = visual_tokens.shape[1]
    D_l = visual_tokens.shape[2]

    # 获取文本 embedding
    text_embeds = token_embedding(input_ids)  # (B, T, D_l)

    all_embeds = []
    for b in range(B):
        # 找到 <image> token 的位置
        image_positions = (input_ids[b] == image_token_id).nonzero(as_tuple=True)[0]

        if len(image_positions) == 0:
            # 纯文本，无图像
            all_embeds.append(text_embeds[b])
            continue

        # 在 <image> 位置插入视觉 token
        img_pos = image_positions[0].item()
        parts = [
            text_embeds[b, :img_pos],          # <image> 之前的文本
            visual_tokens[b],                   # N_vis 个视觉 token
            text_embeds[b, img_pos + 1:],       # <image> 之后的文本
        ]
        all_embeds.append(torch.cat(parts, dim=0))

    # 对齐序列长度 (padding)
    max_len = max(e.shape[0] for e in all_embeds)
    padded_embeds = torch.zeros(B, max_len, D_l, device=visual_tokens.device)
    attention_mask = torch.zeros(B, max_len, device=visual_tokens.device, dtype=torch.long)

    for b, emb in enumerate(all_embeds):
        L = emb.shape[0]
        padded_embeds[b, :L] = emb
        attention_mask[b, :L] = 1

    return padded_embeds, attention_mask

In [ ]:
def create_loss_mask(
    input_ids: torch.Tensor,
    num_visual_tokens: int,
    answer_start_positions: List[int],
    image_token_id: int = 32000,
) -> torch.Tensor:
    """
    创建 loss mask: 只对 assistant 回答的 token 计算 loss。
    
    视觉 token 和 human 问题的位置 mask = 0
    assistant 回答的位置 mask = 1
    """
    B, T = input_ids.shape
    loss_mask = torch.zeros(B, T, device=input_ids.device)

    for b in range(B):
        # <image> 被替换后，序列变长了 num_visual_tokens - 1
        has_image = (input_ids[b] == image_token_id).any()
        offset = (num_visual_tokens - 1) if has_image else 0
        start = answer_start_positions[b] + offset
        loss_mask[b, start:] = 1.0

    return loss_mask

In [ ]:
# 验证多模态拼接
dummy_embedding = nn.Embedding(config.vocab_size + 1, config.llm_hidden_size).to(device)

# 模拟输入: [BOS, <image>, What, is, this, ?, SEP, A, cat, EOS]
#            1    32000   100  101 102  103  2   200 201  2
dummy_ids = torch.tensor([
    [1, 32000, 100, 101, 102, 103, 2, 200, 201, 2],
    [1, 32000, 110, 111, 0,   0,   2, 210, 2,   0],  # shorter, with padding
], device=device)

dummy_vis = torch.randn(2, config.num_patches, config.llm_hidden_size, device=device)

embeds, attn_mask = prepare_multimodal_inputs(dummy_vis, dummy_ids, dummy_embedding)
print(f"Original text length: {dummy_ids.shape[1]}")
print(f"After insertion: {embeds.shape[1]} (added {config.num_patches - 1} visual tokens)")
print(f"Attention mask: {attn_mask[0].sum().item()} valid tokens (sample 0)")
print("✓ 多模态输入拼接验证通过！")

## Part 5：LLM Decoder — LLaMA 风格自回归模型

LLaVA 使用 Vicuna（基于 LLaMA）作为 LLM。我们复用 Week 4 的 LLaMA 架构：

```
LLaMADecoderLayer:
  RMSNorm → Causal Self-Attention (RoPE) → Residual
  → RMSNorm → SwiGLU FFN → Residual
```

教学版简化：使用标准 positional encoding 替代 RoPE，便于理解核心流程。

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm: LLaMA 使用的归一化方式，比 LayerNorm 更高效。"""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class CausalSelfAttention(nn.Module):
    """带 causal mask 的多头自注意力。"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        D = config.llm_hidden_size
        self.num_heads = config.llm_num_heads
        self.head_dim = D // self.num_heads

        self.q_proj = nn.Linear(D, D, bias=False)
        self.k_proj = nn.Linear(D, D, bias=False)
        self.v_proj = nn.Linear(D, D, bias=False)
        self.o_proj = nn.Linear(D, D, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, D = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Causal mask: 上三角为 -inf
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn_weights = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = attn_weights.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        if attention_mask is not None:
            # attention_mask: (B, T) → (B, 1, 1, T)
            pad_mask = (1 - attention_mask.unsqueeze(1).unsqueeze(2).float()) * float('-inf')
            pad_mask = pad_mask.masked_fill(pad_mask != pad_mask, 0.0)  # nan → 0
            attn_weights = attn_weights + pad_mask

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        out = (attn_weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.o_proj(out)


class SwiGLUFFN(nn.Module):
    """SwiGLU FFN: LLaMA 使用的前馈网络。
    
    FFN(x) = (Swish(x · W_gate) ⊙ (x · W_up)) · W_down
    """

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        D = config.llm_hidden_size
        D_ff = config.llm_intermediate_size
        self.gate_proj = nn.Linear(D, D_ff, bias=False)
        self.up_proj = nn.Linear(D, D_ff, bias=False)
        self.down_proj = nn.Linear(D_ff, D, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class LLMDecoderLayer(nn.Module):
    """LLaMA Decoder Layer: RMSNorm + Causal Attn + Residual + RMSNorm + SwiGLU + Residual"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        self.attn_norm = RMSNorm(config.llm_hidden_size, config.rms_norm_eps)
        self.attn = CausalSelfAttention(config)
        self.ffn_norm = RMSNorm(config.llm_hidden_size, config.rms_norm_eps)
        self.ffn = SwiGLUFFN(config)

    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x), attention_mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class LLMDecoder(nn.Module):
    """完整的 LLaMA-style LLM Decoder。"""

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size + 1, config.llm_hidden_size)
        self.layers = nn.ModuleList([LLMDecoderLayer(config) for _ in range(config.llm_num_layers)])
        self.norm = RMSNorm(config.llm_hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.llm_hidden_size, config.vocab_size, bias=False)

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        x = inputs_embeds
        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

    @torch.no_grad()
    def generate(
        self,
        inputs_embeds: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.8,
        top_k: int = 50,
        eos_token_id: int = 2,
    ) -> List[int]:
        """自回归生成：从 inputs_embeds 开始，逐步生成 token。"""
        self.eval()
        generated_ids = []

        for _ in range(max_new_tokens):
            # 截断到 max_seq_len
            if inputs_embeds.shape[1] > self.config.max_seq_len:
                inputs_embeds = inputs_embeds[:, -self.config.max_seq_len:]

            logits = self.forward(inputs_embeds=inputs_embeds)  # (1, T, vocab)
            next_logits = logits[:, -1, :] / temperature         # (1, vocab)

            # Top-K 采样
            if top_k > 0:
                topk_vals, _ = torch.topk(next_logits, top_k)
                next_logits[next_logits < topk_vals[:, -1:]] = float('-inf')

            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # (1, 1)
            token_id = next_token.item()

            if token_id == eos_token_id:
                break

            generated_ids.append(token_id)
            next_embed = self.embed_tokens(next_token)  # (1, 1, D_l)
            inputs_embeds = torch.cat([inputs_embeds, next_embed], dim=1)

        return generated_ids

In [ ]:
# 验证 LLM Decoder
llm = LLMDecoder(config).to(device)
dummy_ids_llm = torch.randint(0, config.vocab_size, (2, 20), device=device)
logits = llm(input_ids=dummy_ids_llm)
print(f"Input:  {dummy_ids_llm.shape}")
print(f"Output: {logits.shape}")
assert logits.shape == (2, 20, config.vocab_size)
print("✓ LLM Decoder shape 验证通过！")

llm_params = sum(p.numel() for p in llm.parameters())
print(f"LLM 参数量: {llm_params:,} ({llm_params/1e6:.1f}M)")

## Part 6：完整 LLaVA 模型 — 三大组件端到端

```
LLaVA 完整前向传播:

Image (B, 3, H, W)
  │  VisionEncoder (冻结)
  ▼
(B, N, D_v)         ← N 个 patch tokens
  │  MLP Projector
  ▼
(B, N, D_l)         ← 视觉 token (LLM 维度)
  │                      Text input_ids (B, T)
  │                           │  Token Embedding
  │                           ▼
  │                      (B, T, D_l)
  │                           │
  ├───── 拼接 ──────────────┤
  ▼
(B, N+T, D_l)       ← 多模态序列
  │  LLM Decoder
  ▼
(B, N+T, vocab_size) ← logits
```

In [ ]:
class LLaVA(nn.Module):
    """
    LLaVA: Large Language and Vision Assistant
    
    三大组件:
    1. VisionEncoder (CLIP ViT, 冻结)
    2. MultimodalProjector (Linear / MLP)
    3. LLMDecoder (LLaMA-style)
    """

    def __init__(self, config: LLaVAConfig):
        super().__init__()
        self.config = config

        # 三大组件
        self.vision_encoder = VisionEncoder(config)
        self.projector = MultimodalProjector(config)
        self.llm = LLMDecoder(config)

        # 冻结 Vision Encoder
        self.freeze_vision_encoder()

    def freeze_vision_encoder(self):
        """冻结 ViT 参数，全程不训练。"""
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

    def freeze_llm(self):
        """Stage 1: 冻结 LLM，只训练 Projector。"""
        for param in self.llm.parameters():
            param.requires_grad = False

    def unfreeze_llm(self):
        """Stage 2: 解冻 LLM，训练 Projector + LLM。"""
        for param in self.llm.parameters():
            param.requires_grad = True

    def get_trainable_params(self) -> dict:
        """统计各组件可训练参数量。"""
        result = {}
        for name, module in [('vision_encoder', self.vision_encoder),
                              ('projector', self.projector),
                              ('llm', self.llm)]:
            total = sum(p.numel() for p in module.parameters())
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            result[name] = {'total': total, 'trainable': trainable}
        return result

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """编码图像: Image → ViT → Projector → (B, N, D_l)"""
        with torch.no_grad():
            vision_features = self.vision_encoder(pixel_values)  # (B, N, D_v)
        visual_tokens = self.projector(vision_features)          # (B, N, D_l)
        return visual_tokens

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        loss_mask: Optional[torch.Tensor] = None,
    ) -> dict:
        """
        LLaVA 前向传播:
        1. 编码图像 → 视觉 token
        2. 拼接视觉 token 和文本 token
        3. 送入 LLM 生成 logits
        4. 计算 loss（仅 assistant 部分）
        """
        # Step 1: 编码图像
        visual_tokens = self.encode_image(pixel_values)  # (B, N, D_l)

        # Step 2: 多模态拼接
        inputs_embeds, attention_mask = prepare_multimodal_inputs(
            visual_tokens, input_ids, self.llm.embed_tokens, self.config.image_token_id
        )

        # Step 3: LLM 前向
        logits = self.llm(inputs_embeds=inputs_embeds, attention_mask=attention_mask)

        result = {'logits': logits}

        # Step 4: 计算 loss（如果提供了 labels）
        if labels is not None:
            # Shift: 预测下一个 token
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, self.config.vocab_size),
                shift_labels.view(-1),
                ignore_index=-100,  # padding 位置不计算 loss
                reduction='none',
            )

            if loss_mask is not None:
                shift_mask = loss_mask[:, 1:].contiguous().view(-1)
                loss = (loss * shift_mask).sum() / shift_mask.sum().clamp(min=1)
            else:
                loss = loss.mean()

            result['loss'] = loss

        return result

In [ ]:
# 验证完整 LLaVA 模型
model = LLaVA(config).to(device)

# 统计参数
params = model.get_trainable_params()
print("=" * 60)
print(f"{'组件':<20} {'总参数':>12} {'可训练':>12} {'冻结':>12}")
print("-" * 60)
for name, info in params.items():
    frozen = info['total'] - info['trainable']
    print(f"{name:<20} {info['total']:>12,} {info['trainable']:>12,} {frozen:>12,}")
total_all = sum(v['total'] for v in params.values())
total_train = sum(v['trainable'] for v in params.values())
print("-" * 60)
print(f"{'合计':<20} {total_all:>12,} {total_train:>12,} {total_all - total_train:>12,}")
print("=" * 60)

In [ ]:
# 前向传播测试
B = 2
dummy_images = torch.randn(B, 3, config.image_size, config.image_size, device=device)
# [BOS, <image>, token, token, ..., SEP, answer_token, ..., EOS]
seq_len = 20
dummy_input_ids = torch.randint(3, config.vocab_size, (B, seq_len), device=device)
dummy_input_ids[:, 0] = config.bos_token_id
dummy_input_ids[:, 1] = config.image_token_id  # <image> 占位符

# Labels: -100 for positions we don't compute loss on
total_len = seq_len + config.num_patches - 1  # <image> replaced by N visual tokens
dummy_labels = torch.full((B, total_len), -100, device=device)
dummy_labels[:, -5:] = torch.randint(3, config.vocab_size, (B, 5), device=device)

outputs = model(dummy_images, dummy_input_ids, labels=dummy_labels)
print(f"Logits shape: {outputs['logits'].shape}")
print(f"Loss: {outputs['loss'].item():.4f}")
print("✓ LLaVA 端到端前向传播验证通过！")

## Part 7：两阶段训练 — 从对齐到指令跟随

### Stage 1: 预训练对齐（Feature Alignment）

| 设置 | 值 |
|------|------|
| 训练参数 | **仅 Projector** |
| 冻结参数 | ViT + LLM |
| 数据 | 图文描述对（caption 数据）|
| 目标 | 让 Projector 学会将视觉特征"翻译"成 LLM 可理解的表示 |
| 学习率 | $2 \times 10^{-3}$ |

### Stage 2: 指令微调（Visual Instruction Tuning）

| 设置 | 值 |
|------|------|
| 训练参数 | **Projector + LLM** |
| 冻结参数 | 仅 ViT |
| 数据 | 多轮视觉对话 + 推理任务 |
| 目标 | 让模型学会多模态指令跟随 |
| 学习率 | $2 \times 10^{-5}$ |

下面使用合成数据模拟两个阶段的训练流程。

In [ ]:
def create_synthetic_batch(
    config: LLaVAConfig,
    batch_size: int = 4,
    text_len: int = 30,
    answer_start: int = 20,
    device: str = 'cpu',
) -> dict:
    """
    创建合成训练数据:
    - 随机图像
    - 随机 token 序列，第 2 个位置是 <image>
    - Labels: 只对 answer 部分有效
    - Loss mask: answer 部分为 1
    """
    images = torch.randn(batch_size, 3, config.image_size, config.image_size, device=device)

    input_ids = torch.randint(3, config.vocab_size, (batch_size, text_len), device=device)
    input_ids[:, 0] = config.bos_token_id
    input_ids[:, 1] = config.image_token_id

    N = config.num_patches
    expanded_len = text_len + N - 1

    labels = torch.full((batch_size, expanded_len), -100, dtype=torch.long, device=device)
    # answer 部分 offset by visual token insertion
    answer_offset = answer_start + N - 1
    for i in range(batch_size):
        ans_len = expanded_len - answer_offset
        if ans_len > 0:
            labels[i, answer_offset:] = torch.randint(3, config.vocab_size, (ans_len,), device=device)

    loss_mask = torch.zeros(batch_size, expanded_len, device=device)
    loss_mask[:, answer_offset:] = 1.0

    return {
        'pixel_values': images,
        'input_ids': input_ids,
        'labels': labels,
        'loss_mask': loss_mask,
    }

In [ ]:
def train_stage(
    model: LLaVA,
    config: LLaVAConfig,
    stage: int,
    num_steps: int = 50,
    batch_size: int = 4,
    lr: float = 2e-3,
    device: str = 'cpu',
):
    """
    执行一个训练阶段。
    
    Stage 1: 冻结 ViT + LLM，只训练 Projector (lr=2e-3)
    Stage 2: 冻结 ViT，训练 Projector + LLM (lr=2e-5)
    """
    print(f"\n{'='*60}")
    print(f"Stage {stage}: {'预训练对齐' if stage == 1 else '指令微调'}")
    print(f"{'='*60}")

    if stage == 1:
        model.freeze_llm()
    else:
        model.unfreeze_llm()
    model.freeze_vision_encoder()  # ViT 始终冻结

    params_info = model.get_trainable_params()
    for name, info in params_info.items():
        status = '训练' if info['trainable'] > 0 else '冻结'
        print(f"  {name}: {status} ({info['trainable']:,} / {info['total']:,})")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)

    model.train()
    losses = []

    for step in range(num_steps):
        batch = create_synthetic_batch(config, batch_size=batch_size, device=device)

        optimizer.zero_grad()
        outputs = model(
            pixel_values=batch['pixel_values'],
            input_ids=batch['input_ids'],
            labels=batch['labels'],
            loss_mask=batch['loss_mask'],
        )
        loss = outputs['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        losses.append(loss.item())
        if (step + 1) % 10 == 0:
            avg_loss = sum(losses[-10:]) / 10
            print(f"  Step {step+1:>3}/{num_steps} | Loss: {loss.item():.4f} | Avg(10): {avg_loss:.4f}")

    print(f"Stage {stage} 完成！最终 loss: {losses[-1]:.4f}")
    return losses

In [ ]:
# === 两阶段训练 ===
model = LLaVA(config).to(device)

# Stage 1: 预训练对齐 — 只训练 Projector
stage1_losses = train_stage(model, config, stage=1, num_steps=50, lr=2e-3, device=device)

# Stage 2: 指令微调 — 训练 Projector + LLM
stage2_losses = train_stage(model, config, stage=2, num_steps=50, lr=2e-5, device=device)

In [ ]:
# 可视化训练 loss 曲线
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(stage1_losses, 'b-', alpha=0.7)
    axes[0].set_title('Stage 1: 预训练对齐 (Projector Only)', fontsize=13)
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(stage2_losses, 'r-', alpha=0.7)
    axes[1].set_title('Stage 2: 指令微调 (Projector + LLM)', fontsize=13)
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print("✓ 训练曲线已生成")
except ImportError:
    print("matplotlib 不可用，跳过可视化")
    print(f"Stage 1 loss: {stage1_losses[0]:.4f} → {stage1_losses[-1]:.4f}")
    print(f"Stage 2 loss: {stage2_losses[0]:.4f} → {stage2_losses[-1]:.4f}")

## Part 8：推理 — 图文问答 Demo

完整推理流程：

```
1. 编码图像: Image → ViT (冻结) → patch tokens (B, N, D_v)
2. 投影:     patch tokens → Projector → visual tokens (B, N, D_l)
3. 编码问题: "What is this?" → Tokenizer → token ids → embeddings
4. 拼接:     [visual_tokens, text_embeddings] → (B, N+T, D_l)
5. 生成:     LLM.generate(inputs_embeds) → token ids
6. 解码:     token ids → Tokenizer.decode → "This is a cat."
```

由于我们使用随机初始化模型和合成数据，生成结果是随机的。
重点是验证**推理流程的正确性**。

In [ ]:
class SimpleTokenizer:
    """简化版 Tokenizer，用于演示推理流程。
    
    实际使用中应替换为 LLaMA tokenizer (sentencepiece)。
    """

    def __init__(self, vocab_size: int = 32000):
        self.vocab_size = vocab_size
        # 简化词表: token_id → word
        self.special_tokens = {
            0: '<pad>', 1: '<bos>', 2: '<eos>',
            32000: '<image>',
        }
        self.demo_vocab = {
            100: 'What', 101: 'is', 102: 'in', 103: 'this',
            104: 'image', 105: '?', 106: 'Describe', 107: 'the',
            108: 'photo', 109: '.', 110: 'A', 111: 'The',
            112: 'cat', 113: 'dog', 114: 'bird', 115: 'car',
            116: 'sitting', 117: 'on', 118: 'a', 119: 'table',
            120: 'with', 121: 'red', 122: 'blue', 123: 'green',
            124: 'sky', 125: 'tree', 126: 'building', 127: 'person',
        }
        self.id_to_word = {**self.special_tokens, **self.demo_vocab}

    def encode(self, text: str) -> List[int]:
        """简化编码：按空格分词，未知词映射为随机 id。"""
        word_to_id = {v: k for k, v in self.demo_vocab.items()}
        tokens = [1]  # BOS
        for word in text.split():
            if word == '<image>':
                tokens.append(32000)
            elif word in word_to_id:
                tokens.append(word_to_id[word])
            else:
                tokens.append(hash(word) % (self.vocab_size - 200) + 200)
        return tokens

    def decode(self, token_ids: List[int]) -> str:
        """将 token id 序列解码为文本。"""
        words = []
        for tid in token_ids:
            if tid in self.id_to_word:
                words.append(self.id_to_word[tid])
            else:
                words.append(f'[{tid}]')
        return ' '.join(words)

In [ ]:
@torch.no_grad()
def llava_inference(
    model: LLaVA,
    image: torch.Tensor,
    question: str,
    tokenizer: SimpleTokenizer,
    max_new_tokens: int = 30,
    temperature: float = 0.8,
) -> str:
    """
    LLaVA 推理: 给定图像和问题，生成自然语言回答。
    
    Steps:
    1. 编码图像 → 视觉 token
    2. 编码问题 → 文本 embedding
    3. 拼接视觉 + 文本
    4. LLM 自回归生成
    5. 解码为文本
    """
    model.eval()
    device = next(model.parameters()).device

    # Step 1: 编码图像
    if image.dim() == 3:
        image = image.unsqueeze(0)  # (3, H, W) → (1, 3, H, W)
    visual_tokens = model.encode_image(image.to(device))  # (1, N, D_l)

    # Step 2: 编码问题
    question_with_image = f"<image> {question}"
    token_ids = tokenizer.encode(question_with_image)
    input_ids = torch.tensor([token_ids], device=device)  # (1, T)

    # Step 3: 拼接
    inputs_embeds, _ = prepare_multimodal_inputs(
        visual_tokens, input_ids, model.llm.embed_tokens, model.config.image_token_id
    )

    # Step 4: 自回归生成
    generated_ids = model.llm.generate(
        inputs_embeds=inputs_embeds,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        eos_token_id=model.config.eos_token_id,
    )

    # Step 5: 解码
    answer = tokenizer.decode(generated_ids)
    return answer

In [ ]:
# === 图文问答 Demo ===
tokenizer = SimpleTokenizer()

# 模拟输入图像
test_image = torch.randn(3, config.image_size, config.image_size, device=device)

questions = [
    "What is in this image ?",
    "Describe the photo .",
    "What is the cat sitting on ?",
]

print("=" * 60)
print("LLaVA 图文问答 Demo")
print("(使用随机初始化模型，输出为随机 token，验证流程正确性)")
print("=" * 60)

for q in questions:
    answer = llava_inference(model, test_image, q, tokenizer)
    print(f"\nQ: {q}")
    print(f"A: {answer}")

print("\n✓ 推理流程验证通过！")

## Part 9：对比验证与总结

### 9.1 与标准 LLaVA 参数量对比

In [ ]:
# 标准 LLaVA 配置（仅用于参数量估算，不实际实例化）
print("=" * 65)
print("参数量对比: 教学版 vs 标准 LLaVA-1.5-7B")
print("=" * 65)

our_params = model.get_trainable_params()

# 标准 LLaVA-1.5-7B 参数量
standard_params = {
    'vision_encoder': {'total': 304_000_000, 'desc': 'CLIP ViT-L/14 (304M)'},
    'projector':      {'total': 8_400_000,   'desc': '2-layer MLP (8.4M)'},
    'llm':            {'total': 6_700_000_000, 'desc': 'Vicuna-7B (6.7B)'},
}

print(f"\n{'组件':<20} {'教学版':>15} {'LLaVA-1.5-7B':>15} {'缩放比':>10}")
print("-" * 65)
for name in ['vision_encoder', 'projector', 'llm']:
    ours = our_params[name]['total']
    std = standard_params[name]['total']
    ratio = std / ours
    print(f"{name:<20} {ours:>15,} {std:>15,} {ratio:>9.0f}x")

our_total = sum(v['total'] for v in our_params.values())
std_total = sum(v['total'] for v in standard_params.values())
print("-" * 65)
print(f"{'合计':<20} {our_total:>15,} {std_total:>15,} {std_total/our_total:>9.0f}x")
print(f"\n教学版总参数: {our_total/1e6:.1f}M")
print(f"标准版总参数: {std_total/1e9:.1f}B")

### 9.2 维度流完整追踪

In [ ]:
print("=" * 65)
print("LLaVA 维度流完整追踪 (教学版配置)")
print("=" * 65)

N = config.num_patches
D_v = config.vision_hidden_size
D_l = config.llm_hidden_size
V = config.vocab_size
T_text = 15  # 示例文本长度

print(f"""
图像 (1, 3, {config.image_size}, {config.image_size})
  │  PatchEmbedding: Conv2d(3, {D_v}, {config.patch_size}, {config.patch_size}) + CLS + PosEmb
  ▼
(1, {N+1}, {D_v})         ← {N} patches + 1 CLS, 每个 {D_v} 维
  │  {config.vision_num_layers} × ViTBlock, 取倒数第二层, 去掉 CLS
  ▼
(1, {N}, {D_v})           ← {N} 个视觉特征
  │  MLP Projector: Linear({D_v}, {D_l}) → GELU → Linear({D_l}, {D_l})
  ▼
(1, {N}, {D_l})           ← {N} 个视觉 token (LLM 维度)

文本 "<image> What is this?" → Tokenizer → [{T_text} tokens]
  │  Token Embedding({V}, {D_l})
  ▼
(1, {T_text}, {D_l})       ← 文本 embedding

拼接: <image> 占位符替换为 {N} 个视觉 token
  ▼
(1, {N + T_text - 1}, {D_l})    ← 多模态序列 ({N} vis + {T_text-1} text)
  │  {config.llm_num_layers} × LLaMADecoderLayer (RMSNorm + CausalAttn + SwiGLU)
  ▼
(1, {N + T_text - 1}, {D_l})
  │  RMSNorm + lm_head: Linear({D_l}, {V})
  ▼
(1, {N + T_text - 1}, {V})   ← logits, 自回归生成
""")

### 9.3 与 HuggingFace LLaVA 的架构对应

In [ ]:
print("=" * 70)
print("我们的实现 vs HuggingFace LLaVA 架构对应关系")
print("=" * 70)

mapping = [
    ("LLaVA",              "LlavaForConditionalGeneration",  "顶层模型"),
    ("VisionEncoder",      "CLIPVisionModel",               "CLIP ViT 视觉编码器"),
    ("├─ PatchEmbedding",  "├─ CLIPVisionEmbeddings",       "Patch + CLS + PosEmb"),
    ("├─ ViTBlock",        "├─ CLIPEncoderLayer",           "LN + MSA + LN + MLP"),
    ("└─ LayerNorm",       "└─ post_layernorm",             "最终归一化"),
    ("MultimodalProjector","LlavaMultiModalProjector",      "视觉→语言投影"),
    ("├─ Linear+GELU",     "├─ linear_1 + act",             "第一层 + 激活"),
    ("└─ Linear",          "└─ linear_2",                   "第二层"),
    ("LLMDecoder",         "LlamaForCausalLM",              "LLaMA 语言模型"),
    ("├─ embed_tokens",    "├─ model.embed_tokens",         "Token Embedding"),
    ("├─ LLMDecoderLayer", "├─ LlamaDecoderLayer",          "Decoder Layer"),
    ("├─ RMSNorm",         "├─ LlamaRMSNorm",               "归一化"),
    ("└─ lm_head",         "└─ lm_head",                    "输出投影"),
]

print(f"\n{'我们的实现':<25} {'HuggingFace':^35} {'说明':>10}")
print("-" * 70)
for ours, hf, desc in mapping:
    print(f"{ours:<25} {hf:<35} {desc}")

print("\n✓ 架构完全对应！区别仅在于教学版使用了更小的维度。")

### 9.4 总结

本 notebook 完整实现了 LLaVA 多模态大模型的核心代码：

| Part | 内容 | 关键代码 |
|------|------|------|
| Part 1 | 模型配置 | `LLaVAConfig` — 统一管理三组件超参数 |
| Part 2 | Vision Encoder | `VisionEncoder` — 复用 ViT，取倒数第二层，去掉 CLS |
| Part 3 | MLP Projector | `MultimodalProjector` — Linear / 2-layer MLP |
| Part 4 | 多模态拼接 | `prepare_multimodal_inputs` — \<image\> 替换 + padding |
| Part 5 | LLM Decoder | `LLMDecoder` — RMSNorm + CausalAttn + SwiGLU |
| Part 6 | 完整 LLaVA | `LLaVA` — 三组件端到端 + 参数冻结策略 |
| Part 7 | 两阶段训练 | Stage 1 对齐 + Stage 2 指令微调 |
| Part 8 | 推理 Demo | `llava_inference` — 图文问答完整流程 |
| Part 9 | 对比验证 | 参数量 + 维度流 + HuggingFace 对应 |

### 面试闭卷手写清单

- [ ] MLP Projector: `nn.Linear(D_v, D_l) → GELU → nn.Linear(D_l, D_l)`
- [ ] 多模态拼接: 找到 `<image>` 位置 → 替换为视觉 token → padding
- [ ] Loss mask: 只对 assistant 回答计算 loss
- [ ] 推理流程: `encode_image → project → concat → generate → decode`
- [ ] 两阶段训练的参数冻结策略

---

### 明日预告

Day 7 将进入**多模态前沿与全周复盘**：
- 音频多模态：Whisper → LLM
- 视频多模态：Video-LLaMA / LLaVA-Video
- 统一多模态模型：Gemini / GPT-4o
- 全周知识图谱串联与面试自检